# Clean features_6

| Step | Action | Rationale |
|------|--------|-----------|
| 0 | **Recovery** | Restore 3 columns corrupted by an accidental early run on 11 pairs |
| 1 | Drop 6 lookahead label columns | `mfe_*` / `trail_*` walk up to 72 future bars |
| 2 | Drop 3 exact duplicate columns | `rv`=`rv_close`; `entropy_norm_delta_6h`=`entropy_change_6h`; `hurst_6h_delta_6h`=`hurst_change` |
| 3 | Clip Kyle lambda (6 cols) | Max z-score up to 105. `kyle_lambda_r2` excluded (R² in [0,1]) |
| 4 | Clip `range_return_ratio` at 10.0 | Fixed cap removes blow-ups (max=252), keeps real distribution |

**NOT changed:** `order_imbalance` kept (differs from `buy_volume_frac` by up to 0.82).

**Output:** 110 columns per pair (119 - 9), overwrites in place.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

FEATURES_DIR   = Path('../backend/data/features_6')
FEATURES_2_DIR = Path('../backend/data/features_2')
FEATURES_3_DIR = Path('../backend/data/features_3')

PAIRS = [
    'AUDJPY', 'AUDNZD', 'AUDUSD', 'CADJPY', 'CHFJPY',
    'EURAUD', 'EURGBP', 'EURJPY', 'EURUSD',
    'GBPJPY', 'GBPUSD', 'NZDUSD',
    'USDCAD', 'USDCHF', 'USDJPY'
]
PAIRS_MAJORS = ['AUDUSD', 'GBPUSD', 'NZDUSD', 'USDCAD', 'USDCHF', 'USDJPY', 'EURUSD']

print(f'Features dir: {FEATURES_DIR.resolve()}')

## Step 0 — Recovery

An accidental run of the old (buggy) notebook corrupted 11 pairs:
- `order_imbalance` — wrongly dropped
- `kyle_lambda_r2` — wrongly clipped (R² truncated at [0.386, 0.775])
- `range_return_ratio` — wrongly clipped at 99.5th pct (0.77)

These 3 columns exist untouched in features_2/3. We restore them first.
The 4 untouched pairs (NZDUSD, USDCAD, USDCHF, USDJPY) are skipped.

In [ ]:
CORRUPTED_PAIRS = [
    'AUDJPY', 'AUDNZD', 'AUDUSD', 'CADJPY', 'CHFJPY',
    'EURAUD', 'EURGBP', 'EURJPY', 'EURUSD', 'GBPJPY', 'GBPUSD'
]
RESTORE_COLS = ['order_imbalance', 'kyle_lambda_r2', 'range_return_ratio']

for pair in CORRUPTED_PAIRS:
    src_path = (
        FEATURES_2_DIR / f'{pair}_microstructure.parquet'
        if pair in PAIRS_MAJORS
        else FEATURES_3_DIR / f'{pair}_microstructure.parquet'
    )
    df_src = pd.read_parquet(src_path, columns=RESTORE_COLS)

    path = FEATURES_DIR / f'{pair}_features.parquet'
    df = pd.read_parquet(path)

    for col in RESTORE_COLS:
        df[col] = df_src[col].reindex(df.index).astype(np.float32)

    df.to_parquet(path)
    print(f'{pair}: restored {RESTORE_COLS}')

df_v = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
print(f'\nEURUSD shape: {df_v.shape} (should be 119)')
print(f'order_imbalance present: {"order_imbalance" in df_v.columns}')
print(f'kyle_lambda_r2 max: {df_v["kyle_lambda_r2"].max():.4f} (should be ~1.0)')
print(f'range_return_ratio max: {df_v["range_return_ratio"].max():.2f} (should be ~252)')


## Step 1 — Define columns to drop

In [ ]:
# Lookahead: computed by walking forward up to 72 future bars
LOOKAHEAD_COLS = [
    'mfe_long_pips', 'mfe_short_pips',
    'trail_long_bars', 'trail_short_bars',
    'trail_stop_pips', 'mfe_atr_24',
]

# Exact duplicates (max_diff <= 6e-8, float32 rounding only)
# order_imbalance NOT here -- differs from buy_volume_frac by up to 0.82
DUPLICATE_COLS = [
    'rv',                    # == rv_close (max_diff=0)
    'entropy_norm_delta_6h', # == entropy_change_6h (max_diff=6e-8)
    'hurst_6h_delta_6h',     # == hurst_change (max_diff=6e-8)
]

COLS_TO_DROP = LOOKAHEAD_COLS + DUPLICATE_COLS
print(f'Total to drop: {len(COLS_TO_DROP)}')
print(f'  Lookahead ({len(LOOKAHEAD_COLS)}): {LOOKAHEAD_COLS}')
print(f'  Duplicates ({len(DUPLICATE_COLS)}): {DUPLICATE_COLS}')


## Step 2 — Verify on EURUSD before running

In [ ]:
df = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
print(f'EURUSD shape: {df.shape} (should be 119)')

print(f'Lookahead cols found: {[c for c in LOOKAHEAD_COLS if c in df.columns]}')

print('\nDuplicate verification:')
for drop_col, keep_col in [('rv','rv_close'), ('entropy_norm_delta_6h','entropy_change_6h'), ('hurst_6h_delta_6h','hurst_change')]:
    corr = df[[drop_col, keep_col]].corr().iloc[0,1]
    max_diff = (df[drop_col] - df[keep_col]).abs().max()
    print(f'  {drop_col} vs {keep_col}: corr={corr:.4f}, max_diff={max_diff:.2e}')

KYLE_CLIP_COLS = ['kyle_lambda', 'kyle_lambda_ma12', 'kyle_lambda_change',
                  'kyle_lambda_delta_3h', 'kyle_lambda_delta_6h', 'kyle_lambda_delta_12h']
print('\nKyle clip preview (kyle_lambda_r2 excluded):')
for col in KYLE_CLIP_COLS:
    lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
    n = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'  {col}: [{lo:.2e}, {hi:.2e}], rows={n}')

n_rrr = (df['range_return_ratio'] > 10.0).sum()
print(f'\nrange_return_ratio: cap=10.0, rows clipped={n_rrr} ({100*n_rrr/len(df):.3f}%)')
print(f'kyle_lambda_r2 max (should be ~1.0): {df["kyle_lambda_r2"].max():.4f}')


## Step 3 — Clean all pairs

In [ ]:
KYLE_CLIP_COLS = ['kyle_lambda', 'kyle_lambda_ma12', 'kyle_lambda_change',
                  'kyle_lambda_delta_3h', 'kyle_lambda_delta_6h', 'kyle_lambda_delta_12h']
# kyle_lambda_r2 is R^2 in [0,1] -- NOT clipped

summary = []

for pair in PAIRS:
    path = FEATURES_DIR / f'{pair}_features.parquet'
    df = pd.read_parquet(path)
    shape_before = df.shape

    # 1. Drop lookahead + duplicates
    df = df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns])

    # 2. Clip kyle lambda at [1st, 99th] pct (r2 excluded)
    for col in [c for c in KYLE_CLIP_COLS if c in df.columns]:
        lo, hi = df[col].quantile(0.01), df[col].quantile(0.99)
        df[col] = df[col].clip(lower=lo, upper=hi)

    # 3. Clip range_return_ratio at fixed cap 10.0
    if 'range_return_ratio' in df.columns:
        df['range_return_ratio'] = df['range_return_ratio'].clip(upper=10.0)

    df.to_parquet(path)
    shape_after = df.shape
    summary.append({'pair': pair, 'cols_before': shape_before[1], 'cols_after': shape_after[1], 'rows': shape_after[0]})
    print(f'{pair}: {shape_before} -> {shape_after}')

print('\nDone.')


## Step 4 — Verify output

In [ ]:
print(pd.DataFrame(summary).to_string(index=False))

df_check = pd.read_parquet(FEATURES_DIR / 'EURUSD_features.parquet')
print(f'\nEURUSD post-clean shape: {df_check.shape} (expected (100675, 110))')

remaining_lookahead = [c for c in LOOKAHEAD_COLS if c in df_check.columns]
remaining_dupes     = [c for c in DUPLICATE_COLS if c in df_check.columns]
print(f'Lookahead remaining: {remaining_lookahead or "none"}')
print(f'Duplicates remaining: {remaining_dupes or "none"}')

print('\nExpected kept columns:')
for col in ['rv_close', 'entropy_change_6h', 'hurst_change', 'buy_volume_frac', 'order_imbalance', 'kyle_lambda_r2']:
    print(f'  {col}: {"OK" if col in df_check.columns else "MISSING"}')

print(f'\nkyle_lambda_r2 max: {df_check["kyle_lambda_r2"].max():.4f} (should be ~1.0, not 0.775)')
print(f'range_return_ratio max: {df_check["range_return_ratio"].max():.4f} (should be ~10.0)')
